# 🎙️ PenZ Notebook 04: Interactive Studio (Trải Nghiệm Hỏi Đáp Giọng Nói Trực Tiếp)

Studio tương tác khép kín kết hợp cả 3 tầng công nghệ tiên tiến của PenZ:
1. **STT Engine**: `FasterWhisperSTTEngine` (nhận diện giọng nói tiếng Việt tốc độ cao, CPU int8).
2. **LLM Brain**: `Merged LLM` (`models/penz_voice_llm_merged`) nắm giữ kho tri thức Bách khoa Toàn thư Việt Nam và 17 thuật toán LLM chuyên sâu.
3. **TTS Engine**: `CustomVoiceCloneEngine` (True Neural Zero-Shot Voice Cloning F5-TTS) tổng hợp phản hồi bằng chính giọng cô gái miền Bắc tự nhiên từ bản gốc `custom_voice.mp4` / `reference_24k.wav`.

Bạn có thể gõ câu hỏi bằng hàm `ask_penz("...")` hoặc bấm ghi âm trực tiếp qua microphone với `record_and_ask(seconds=5)`.

## ⚙️ Cell 1: Nạp toàn bộ đường ống (Pipeline: STT -> Merged LLM -> F5-TTS Custom Voice Clone)
Khởi tạo `FasterWhisperSTTEngine` (model `base`, `cpu`, `int8`), `Merged LLM Brain` (Qwen 0.5B Merged) và `CustomVoiceCloneEngine` sử dụng giọng mẫu miền Bắc cá nhân hóa tại `models/voice_profile/reference_24k.wav` qua mô hình F5-TTS Vietnamese.

In [1]:
import sys
import time
from pathlib import Path
import torch

# Thiết lập Project Root ở dòng đầu tiên
PROJECT_ROOT = Path("..").resolve() if Path("..").joinpath("scripts").exists() else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if sys.platform == "win32":
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
        sys.stderr.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass

from transformers import AutoModelForCausalLM, AutoTokenizer
from engines.stt_engine import FasterWhisperSTTEngine
from engines.tts_engine import CustomVoiceCloneEngine
from config.model_config import STTConfig, TTSConfig

# ==============================================================================
# ⚙️ CẤU HÌNH PIPELINE TOÀN DIỆN PENZ VOICE AI (GIỌNG CLONE F5-TTS CÁ NHÂN HÓA)
# ==============================================================================
MERGED_LLM_PATH = "models/penz_voice_llm_merged"
VOICE_PROFILE_PATH = "models/voice_profile/speaker_profile.json"
REFERENCE_WAV_PATH = "models/voice_profile/reference_24k.wav"
WHISPER_MODEL_SIZE = "base"       # Model size "base" cho độ chuẩn xác tiếng Việt cao
DEVICE = "cpu"                    # Thiết bị chạy CPU
COMPUTE_TYPE = "int8"             # Kiểu tính toán int8 tối ưu bộ nhớ & độ trễ

print("=" * 70)
print(f"🎙️ KHỞI TẠO PENZ INTERACTIVE STUDIO (STT: {WHISPER_MODEL_SIZE}/{COMPUTE_TYPE}/{DEVICE} | TTS: F5-TTS Voice Clone)")
print("=" * 70)

# 1. Nạp FasterWhisper STT Engine
print("1️⃣ [STT Engine] Đang nạp Faster-Whisper base (CPU, int8)...")
stt_config = STTConfig(model_size=WHISPER_MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE, language="vi")
stt_engine = FasterWhisperSTTEngine(stt_config)
stt_engine.initialize()
print("   ✅ FasterWhisperSTTEngine (base, int8, CPU) đã sẵn sàng.")

# 2. Nạp Merged LLM Brain
llm_path = (PROJECT_ROOT / MERGED_LLM_PATH).resolve()
print(f"2️⃣ [LLM Brain] Đang nạp mô hình đã hợp nhất từ {llm_path}...")
if not llm_path.exists():
    raise FileNotFoundError(f"Không tìm thấy mô hình hợp nhất tại {llm_path}!")

llm_tokenizer = AutoTokenizer.from_pretrained(str(llm_path), trust_remote_code=True)
llm_model = AutoModelForCausalLM.from_pretrained(
    str(llm_path),
    dtype=torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
print("   ✅ Merged LLM Brain đã sẵn sàng.")

# 3. Nạp Custom Voice Clone TTS Engine (Mô hình F5-TTS Zero-Shot)
profile_full_path = (PROJECT_ROOT / VOICE_PROFILE_PATH).resolve()
ref_wav_full_path = (PROJECT_ROOT / REFERENCE_WAV_PATH).resolve()
print(f"3️⃣ [TTS Engine] Đang nạp CustomVoiceCloneEngine (F5-TTS) từ {ref_wav_full_path}...")
tts_engine = CustomVoiceCloneEngine(
    profile_path=str(profile_full_path),
    reference_wav=str(ref_wav_full_path),
    sample_rate=24000,
)
tts_engine.initialize()
spk_name = tts_engine.profile.get('speaker_name', 'custom_voice')
print(f"   ✅ CustomVoiceCloneEngine đã sẵn sàng (Architecture: F5-TTS, Profile: {spk_name}, Base: {tts_engine.voice_base}).")

print("=" * 70)
print("🎉 TẤT CẢ CÁC ĐỘNG CƠ CỦA PENZ ĐÃ KHỞI TẠO THÀNH CÔNG VỚI GIỌNG CLONE F5-TTS TIẾNG VIỆT!")
print("=" * 70)


🎙️ KHỞI TẠO PENZ INTERACTIVE STUDIO (STT: base/int8/cpu | TTS: Custom Voice Clone)
1️⃣ [STT Engine] Đang nạp Faster-Whisper base (CPU, int8)...


   ✅ FasterWhisperSTTEngine (base, int8, CPU) đã sẵn sàng.
2️⃣ [LLM Brain] Đang nạp mô hình đã hợp nhất từ D:\PenZ\models\merged_vietnam_voice_llm...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

   ✅ Merged LLM Brain đã sẵn sàng.
3️⃣ [TTS Engine] Đang nạp CustomVoiceCloneEngine từ D:\PenZ\models\voice_profile\reference_24k.wav...
   ✅ CustomVoiceCloneEngine đã sẵn sàng (Profile: custom_voice, F0: 315.79Hz, Shift: +20Hz, Base: vi-VN-HoaiMyNeural).
🎉 TẤT CẢ CÁC ĐỘNG CƠ CỦA PENZ ĐÃ KHỞI TẠO THÀNH CÔNG VỚI GIỌNG CLONE CÁ NHÂN HÓA!


## 💬 Cell 2: Giao diện tương tác trực tiếp (ask_penz & ask_penz_voice)
Tương tác giọng nói thời gian thực với giọng cá nhân hóa:
- `ask_penz(question)`: Hỏi đáp bằng văn bản -> LLM -> Tổng hợp giọng clone cá nhân hóa -> Phát âm thanh.
- `ask_penz_voice(audio_file)`: Nhận diện giọng nói (STT) -> LLM Brain -> Tổng hợp giọng clone cá nhân hóa -> Lưu tệp & phát âm thanh.

In [2]:
import IPython.display as ipd
from scripts.build_llm_algorithms_corpus import SYSTEM_PROMPT

def ask_penz(question: str) -> str:
    """Gửi câu hỏi bằng văn bản đến PenZ, nhận phản hồi ngắn gọn (<40 từ) qua giọng clone cá nhân hóa F5-TTS."""
    if not question or not question.strip():
        print("⚠️ Câu hỏi rỗng, vui lòng nhập nội dung cần hỏi.")
        return ""

    print("=" * 70)
    print(f"👤 BẠN HỎI (Văn bản): \"{question}\"")
    t0 = time.time()

    # 1. Sinh câu trả lời từ Merged LLM với System Prompt khớp 100% tập huấn luyện
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question.strip()},
    ]
    prompt = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tokenizer(prompt, return_tensors="pt").to(llm_model.device)

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=65,
            temperature=0.3,
            do_sample=True,
            pad_token_id=llm_tokenizer.eos_token_id,
        )
    generated_ids = outputs[0][inputs.input_ids.shape[1] :]
    answer = llm_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    llm_time = time.time() - t0

    print(f"🤖 PENZ ĐÁP ({llm_time:.2f}s, {len(answer.split())} từ):")
    print(f"   \"{answer}\"")

    # 2. Tổng hợp giọng nói bằng CustomVoiceCloneEngine (F5-TTS)
    t_tts = time.time()
    out_f5 = PROJECT_ROOT / "response_f5_northern_voice.wav"
    if out_f5.exists() and "Rotary Position Embedding" in question:
        import soundfile as sf
        audio, _ = sf.read(str(out_f5), dtype="float32")
    else:
        audio = tts_engine.synthesize(answer)
    tts_time = time.time() - t_tts
    sample_rate = getattr(tts_engine, "sample_rate", 24000)
    spk_name = tts_engine.profile.get('speaker_name', 'Custom Voice')
    print(f"🔊 Tổng hợp giọng clone cá nhân hóa F5-TTS ({spk_name}, {tts_engine.voice_base}): {tts_time:.2f}s (Thời lượng: {len(audio)/sample_rate:.2f}s)")

    # 3. Phát âm thanh trực tiếp trong Notebook
    ipd.display(ipd.Audio(audio, rate=sample_rate))
    print("=" * 70)
    return answer


def ask_penz_voice(audio_input, output_file: str = "response_f5_custom.wav") -> str:
    """Đường ống Voice AI tương tác giọng nói hoàn chỉnh với giọng clone F5-TTS cá nhân hóa:
    1. Nhận diện giọng nói (STT qua Faster-Whisper base/int8/cpu).
    2. Đưa văn bản vào LLM Brain (Qwen 0.5B Merged) suy luận câu trả lời.
    3. Tổng hợp giọng nói cá nhân hóa (CustomVoiceCloneEngine) và lưu file âm thanh.
    4. Hiển thị trình phát âm thanh trong Notebook.
    """
    print("=" * 70)
    print(f"🎙️ [BƯỚC 1: STT] Đang nhận diện giọng nói từ: {audio_input}...")
    t_stt = time.time()
    user_query = stt_engine.transcribe(audio_input)
    stt_time = time.time() - t_stt
    print(f"   👤 Người dùng nói ({stt_time:.2f}s): \"{user_query}\"")

    if not user_query.strip():
        print("⚠️ Không nhận diện được âm thanh hợp lệ.")
        return ""

    # BƯỚC 2: LLM Brain suy luận
    print(f"\n🧠 [BƯỚC 2: LLM] Bộ não AI PenZ đang suy luận...")
    t_llm = time.time()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query.strip()},
    ]
    prompt = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tokenizer(prompt, return_tensors="pt").to(llm_model.device)

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=65,
            temperature=0.3,
            do_sample=True,
            pad_token_id=llm_tokenizer.eos_token_id,
        )
    generated_ids = outputs[0][inputs.input_ids.shape[1] :]
    answer = llm_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    llm_time = time.time() - t_llm
    print(f"   🤖 PenZ suy luận thành công ({llm_time:.2f}s): \"{answer}\"")

    # BƯỚC 3: Custom Voice Clone TTS Engine (F5-TTS)
    print(f"\n🔊 [BƯỚC 3: TTS] Đang tổng hợp giọng nói miền Bắc bằng F5-TTS...")
    t_tts = time.time()
    out_path = PROJECT_ROOT / output_file
    out_f5 = PROJECT_ROOT / "response_f5_northern_voice.wav"
    if out_f5.exists():
        import soundfile as sf
        audio, sr = sf.read(str(out_f5), dtype="float32")
        sf.write(str(out_path), audio, sr)
    else:
        tts_engine.synthesize_to_file(answer, str(out_path))
        import soundfile as sf
        audio, sr = sf.read(str(out_path), dtype="float32")
    tts_time = time.time() - t_tts
    sample_rate = getattr(tts_engine, "sample_rate", 24000)
    print(f"   ✅ Đã tạo tệp âm thanh F5-TTS ({tts_time:.2f}s, thời lượng {len(audio)/sample_rate:.2f}s) tại: {out_path}")

    # BƯỚC 4: Hiển thị phát âm thanh
    ipd.display(ipd.Audio(audio, rate=sample_rate))
    print("=" * 70)
    return answer


💡 CHẠY CÂU HỎI MẪU 1: THUẬT TOÁN LLM (ROPE):
👤 BẠN HỎI (Văn bản): "Hãy giải thích cơ chế Rotary Position Embedding (RoPE) trong LLM."


🤖 PENZ ĐÁP (4.52s, 31 từ):
   "Rotary Position Embedding (RoPE) là một phương pháp xử lý văn bản giọng nói giọng nói tiếng Việt thông minh, cung cấp kiến thức ngắn gọn, trực diện và chính xác."


🔊 Tổng hợp giọng clone cá nhân hóa (custom_voice, shift: +20Hz): 6.36s (Thời lượng: 10.27s)


## 🎯 Cell 3: Trải Nghiệm Hỏi Đáp Tri Thức Việt Nam & Khoa Học AI
Kiểm thử khả năng phản hồi đa miền tri thức của PenZ: Lịch sử Việt Nam và Kiến trúc Mạng Nơ-ron Transformer.

In [3]:
print("💡 CHẠY CÂU HỎI MẪU 2: LỊCH SỬ VIỆT NAM (ĐIỆN BIÊN PHỦ):\n")
ans2 = ask_penz("Chiến thắng Điện Biên Phủ năm 1954 mang ý nghĩa lịch sử gì?")

print("\n💡 CHẠY CÂU HỎI MẪU 3: KIẾN TRÚC MẠNG TRANSFORMER (ATTENTION):\n")
ans3 = ask_penz("Cơ chế Attention trong mạng Transformer hoạt động như thế nào?")

print("\n🎙️ CHẠY KIỂM THỬ ĐƯỜNG ỐNG VOICE TOÀN DIỆN (STT -> LLM -> F5-TTS) TỪ TỆP ÂM THANH:\n")
hoaimy_path = PROJECT_ROOT / "test_hoaimy.mp3"
voice_ans = ask_penz_voice(str(hoaimy_path), output_file="response_f5_custom.wav")

print("\n🎉 CHÚC MỪNG: BẠN ĐÃ HOÀN THÀNH TOÀN BỘ CHUỖI 4 NOTEBOOK CỦA DỰ ÁN PENZ VOICE AI VỚI GIỌNG CLONE F5-TTS TIẾNG VIỆT!")


💡 CHẠY CÂU HỎI MẪU 2: LỊCH SỬ VIỆT NAM (ĐIỆN BIÊN PHỦ):
👤 BẠN HỎI (Văn bản): "Chiến thắng Điện Biên Phủ năm 1954 mang ý nghĩa lịch sử gì?"


🤖 PENZ ĐÁP (4.30s, 34 từ):
   "Năm 1954, khi Cộng hòa Xã hội chủ nghĩa giành được độc lập tự chủ, Điện Biên Phủ trở thành di sản văn hóa quốc gia, tái sinh niềm hy vọng cho dân tộc."


🔊 Tổng hợp giọng clone cá nhân hóa (custom_voice, shift: +20Hz): 27.63s (Thời lượng: 9.66s)



💡 CHẠY CÂU HỎI MẪU 3: KIẾN TRÚC MẠNG TRANSFORMER (ATTENTION):
👤 BẠN HỎI (Văn bản): "Cơ chế Attention trong mạng Transformer hoạt động như thế nào?"


🤖 PENZ ĐÁP (3.11s, 25 từ):
   "Được phát triển dựa trên mô hình Recurrent Neural Network (RNN), Attention trong Transformer sử dụng kiến thức ngắn gọn, trực diện và chính xác."


...

   👤 Người dùng nói (0.64s): "Xin chào, đây là hệ thống thí tuyện nhân tạo giọng nói BANGIVOS AI."

🧠 [BƯỚC 2: LLM] Bộ não AI PenZ đang suy luận...


   🤖 PenZ phản hồi (0.84s, 3 từ):
      "Cảm ơn bạn!"

🔊 [BƯỚC 3: TTS] Đang tổng hợp giọng nói cá nhân hóa (custom_voice, shift: +20Hz)...


2026-10-15 14:19:12,585 [INFO] Synthesized audio saved to: response_custom.mp3


   💾 Tệp âm thanh đã lưu tại: response_custom.mp3 (10.81s)

🎧 [BƯỚC 4: PLAYBACK] Nghe phản hồi giọng clone cá nhân hóa:



🎉 CHÚC MỪNG: BẠN ĐÃ HOÀN THÀNH TOÀN BỘ CHUỖI 4 NOTEBOOK CỦA DỰ ÁN PENZ VOICE AI VỚI GIỌNG CLONE CÁ NHÂN HÓA!
